<a href="https://colab.research.google.com/github/Rajeraghav/AI-Engineer-Journey/blob/main/Day10/LSTM_IMDB_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# LSTM IMPLEMENTATION USING IMDB DATASET
# CSV FILE AS RUNTIME INPUT
# ============================================================

# Step 1: Import required libraries
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout


# ============================================================
# Step 2: Upload IMDB CSV at runtime
# ============================================================

uploaded = files.upload()

# Automatically get uploaded filename
filename = next(iter(uploaded))

print("Uploaded file:", filename)


# ============================================================
# Step 3: Read entire CSV file
# ============================================================

df = pd.read_csv(filename)

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# Step 4: Detect text and sentiment columns
# ============================================================

# Change these two names if your CSV uses different column names

text_column = "review"
label_column = "sentiment"


# ============================================================
# Step 5: Remove missing values
# ============================================================

df = df[[text_column, label_column]].dropna()

print("\nDataset after removing missing values:")
print(df.shape)


# ============================================================
# Step 6: Convert sentiment into numerical labels
# ============================================================

df[label_column] = df[label_column].astype(str).str.lower()

df[label_column] = df[label_column].map({
    "positive": 1,
    "negative": 0
})

# Remove rows that could not be mapped
df = df.dropna(subset=[label_column])

df[label_column] = df[label_column].astype(int)

print("\nClass distribution:")
print(df[label_column].value_counts())


# ============================================================
# Step 7: Separate input and output
# ============================================================

X = df[text_column].astype(str).values
y = df[label_column].values


# ============================================================
# Step 8: Train-Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# ============================================================
# Step 9: Tokenization
# ============================================================

vocab_size = 10000

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)


# ============================================================
# Step 10: Padding
# ============================================================

max_length = 200

X_train_padded = pad_sequences(
    X_train_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test_padded = pad_sequences(
    X_test_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

print("\nTraining data shape:", X_train_padded.shape)
print("Testing data shape:", X_test_padded.shape)


# ============================================================
# Step 11: Build LSTM Model
# ============================================================

embedding_dim = 128

model = Sequential([

    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_length
    ),

    LSTM(128),

    Dropout(0.5),

    Dense(64, activation="relu"),

    Dropout(0.5),

    Dense(1, activation="sigmoid")
])


# ============================================================
# Step 12: Compile Model
# ============================================================

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# ============================================================
# Step 13: Display Model Architecture
# ============================================================

model.summary()


# ============================================================
# Step 14: Train LSTM
# ============================================================

history = model.fit(
    X_train_padded,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)


# ============================================================
# Step 15: Evaluate Model
# ============================================================

test_loss, test_accuracy = model.evaluate(
    X_test_padded,
    y_test,
    verbose=0
)

print("\nTest Loss:", test_loss)
print("Test Accuracy:", test_accuracy)


# ============================================================
# Step 16: Predictions
# ============================================================

y_probability = model.predict(X_test_padded)

y_pred = (y_probability >= 0.5).astype(int).flatten()


# ============================================================
# Step 17: Accuracy
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", accuracy)


# ============================================================
# Step 18: Classification Report
# ============================================================

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Negative", "Positive"]
    )
)


# ============================================================
# Step 19: Confusion Matrix
# ============================================================

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


# ============================================================
# Step 20: Test with a New Review
# ============================================================

new_review = [
    "This movie was absolutely wonderful and I really enjoyed it"
]

new_sequence = tokenizer.texts_to_sequences(new_review)

new_padded = pad_sequences(
    new_sequence,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

prediction = model.predict(new_padded)[0][0]

if prediction >= 0.5:
    print("\nPrediction: Positive")
    print("Probability:", prediction)
else:
    print("\nPrediction: Negative")
    print("Probability:", 1 - prediction)

Saving IMDB-Dataset.csv to IMDB-Dataset.csv
Uploaded file: IMDB-Dataset.csv
Dataset Shape: (50000, 2)

Columns:
['review', 'sentiment']

First 5 rows:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive



Dataset after removing missing values:
(50000, 2)

Class distribution:
sentiment
1    25000
0    25000
Name: count, dtype: int64

Training samples: 40000
Testing samples: 10000

Training data shape: (40000, 200)
Testing data shape: (10000, 200)


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 151s 298ms/step - accuracy: 0.5167 - loss: 0.6936 - val_accuracy: 0.4984 - val_loss: 0.6916
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 203s 301ms/step - accuracy: 0.4993 - loss: 0.6935 - val_accuracy: 0.4989 - val_loss: 0.6928
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 151s 301ms/step - accuracy: 0.5036 - loss: 0.6922 - val_accuracy: 0.4989 - val_loss: 0.6904
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 202s 300ms/step - accuracy: 0.5128 - loss: 0.6904 - val_accuracy: 0.5460 - val_loss: 0.6869
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 201s 298ms/step - accuracy: 0.5201 - loss: 0.6906 - val_accuracy: 0.5201 - val_loss: 0.6905

Test Loss: 0.690340518951416
Test Accuracy: 0.5169000029563904
313/313 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step

Accuracy: 0.5169

Classification Report:
              precision    recall  f1-score   support

    Negative       0.53      0.32      0.40      5000
    Positive       0.51      0.71      0.60      5000

    accuracy                    